In [0]:
# Uppercase station names in stations_cleaned for the join
df_station_zones = spark.table("stations_cleaned") \
    .withColumn("station_name", F.upper(F.col("station_name"))) \
    .select("station_name", "zone") \
    .dropDuplicates(["station_name"])

# Verify the match now
weather_names = [
    r.station_name for r in
    spark.table("weather_history")
    .select("station_name").distinct().collect()
]

matched = df_station_zones \
    .filter(F.col("station_name").isin(weather_names)) \
    .collect()

print(f"✅ Matched {len(matched)} of {len(weather_names)} stations")
for r in matched:
    print(f"  {r.station_name} → {r.zone}")

In [0]:
# See exactly what names exist in both tables
print("=== WEATHER TABLE station names ===")
spark.table("weather_history") \
    .select("station_name").distinct() \
    .show(truncate=False)

print("=== STATIONS_CLEANED sample names ===")
spark.table("stations_cleaned") \
    .select("station_name").distinct() \
    .limit(20) \
    .show(truncate=False)

In [0]:
# Deep diagnostic — show exact characters in both tables
import re

weather_names = [
    r.station_name for r in
    spark.table("weather_history")
    .select("station_name").distinct().collect()
]

station_names = [
    r.station_name for r in
    spark.table("stations_cleaned")
    .select("station_name").distinct()
    .limit(50).collect()
]

print("=== WEATHER names (repr shows hidden chars) ===")
for n in weather_names:
    print(repr(n))

print("\n=== STATIONS_CLEANED names (repr) ===")
for n in station_names[:20]:
    print(repr(n))

print("\n=== Manual match attempt ===")
weather_upper = [n.strip().upper() for n in weather_names]
station_upper = [n.strip().upper() for n in station_names]

for w in weather_upper:
    for s in station_upper:
        if w == s:
            print(f"MATCH: '{w}'")
            break
    else:
        print(f"NO MATCH: '{w}'")

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import create_map, lit
from itertools import chain

zone_name_map = {
    "CR":   "Central Railway",
    "ER":   "Eastern Railway",
    "ECR":  "East Central Railway",
    "ECoR": "East Coast Railway",
    "KR":   "Konkan Railway",
    "NCR":  "North Central Railway",
    "NER":  "North Eastern Railway",
    "NFR":  "Northeast Frontier Railway",
    "NWR":  "North Western Railway",
    "NR":   "Northern Railway",
    "SCR":  "South Central Railway",
    "SER":  "South Eastern Railway",
    "SECR": "South East Central Railway",
    "SWR":  "South Western Railway",
    "SR":   "Southern Railway",
    "WCR":  "West Central Railway",
    "WR":   "Western Railway",
    "MR":   "Metro Railway Kolkata",
}

mapping_expr = create_map([lit(x) for x in chain(*zone_name_map.items())])
print("✅ mapping_expr ready")

In [0]:
# Hardcode zone mapping for the 10 weather stations
# No join needed — we know exactly which zone each belongs to
from pyspark.sql import functions as F

weather_zone_map = {
    "NEW DELHI":                  "NR",    # Northern Railway
    "AGRA CANTT":                 "NCR",   # North Central Railway
    "PT DEEN DAYAL UPADHYAY JN": "ECR",   # East Central Railway
    "HOWRAH JN":                  "ER",    # Eastern Railway
    "KALYAN JN":                  "CR",    # Central Railway
    "VIJAYAWADA JN":              "SCR",   # South Central Railway
    "CHENNAI CENTRAL":            "SR",    # Southern Railway
    "ARAKKONAM":                  "SR",    # Southern Railway
    "KATPADI JN":                 "SR",    # Southern Railway
    "SALEM JN":                   "SR",    # Southern Railway
}

# Build a small DataFrame from the hardcoded map
weather_zone_rows = [(k, v) for k, v in weather_zone_map.items()]
df_weather_zone_lookup = spark.createDataFrame(
    weather_zone_rows,
    ["station_name", "zone"]
)

# Join weather_history to this lookup
df_weather_zoned = spark.table("weather_history") \
    .join(df_weather_zone_lookup, on="station_name", how="left") \
    .withColumn("Zonal_Railway", mapping_expr[F.col("zone")]) \
    .filter(F.col("Zonal_Railway").isNotNull())

# Average precipitation per zone
df_weather_feat = df_weather_zoned.groupBy("Zonal_Railway").agg(
    F.avg("precipitation_mm").alias("avg_precip_mm")
)

# Normalize to 0-1
max_precip = df_weather_feat.agg(F.max("avg_precip_mm")).first()[0]
df_weather_feat = df_weather_feat.withColumn(
    "weather_correlation",
    F.round(F.col("avg_precip_mm") / max_precip, 4)
).select("Zonal_Railway", "weather_correlation")

print(f"✅ weather_correlation computed for {df_weather_feat.count()} zones")
df_weather_feat.orderBy("weather_correlation", ascending=False).show(truncate=False)

In [0]:
# Join running_history to stations_cleaned to get zone
df_delays = spark.table("running_history") \
    .join(
        spark.table("stations_cleaned").select("station_code", "zone"),
        on="station_code",
        how="inner"
    )

# Map short zone codes to full names
df_delays = df_delays.withColumn(
    "Zonal_Railway",
    mapping_expr[F.col("zone")]
).filter(F.col("Zonal_Railway").isNotNull())

# Average significant delay % per zone
df_delay_feat = df_delays.groupBy("Zonal_Railway").agg(
    F.avg("pct_significant_delay").alias("avg_sig_delay")
)

# Normalize to 0-1
max_delay = df_delay_feat.agg(F.max("avg_sig_delay")).first()[0]
df_delay_feat = df_delay_feat.withColumn(
    "delay_spike_freq",
    F.round(F.col("avg_sig_delay") / max_delay, 4)
).select("Zonal_Railway", "delay_spike_freq")

print(f"✅ delay_spike_freq computed for {df_delay_feat.count()} zones")
df_delay_feat.orderBy("delay_spike_freq", ascending=False).show(truncate=False)

In [0]:
all_zones = [
    ("Central Railway",),
    ("Eastern Railway",),
    ("East Central Railway",),
    ("East Coast Railway",),
    ("Konkan Railway",),
    ("North Central Railway",),
    ("North Eastern Railway",),
    ("Northeast Frontier Railway",),
    ("North Western Railway",),
    ("Northern Railway",),
    ("South Central Railway",),
    ("South Eastern Railway",),
    ("South East Central Railway",),
    ("South Western Railway",),
    ("Southern Railway",),
    ("West Central Railway",),
    ("Western Railway",),
    ("Metro Railway Kolkata",),
    ("Production Units",),
]

df_all_zones = spark.createDataFrame(all_zones, ["Zonal_Railway"])

# Cast both to Double explicitly to avoid type mismatch
df_delay_feat_cast = df_delay_feat.withColumn(
    "delay_spike_freq", F.col("delay_spike_freq").cast("double")
)
df_weather_feat_cast = df_weather_feat.withColumn(
    "weather_correlation", F.col("weather_correlation").cast("double")
)

# Left join real features onto full zone list
df_features = df_all_zones \
    .join(df_delay_feat_cast, on="Zonal_Railway", how="left") \
    .join(df_weather_feat_cast, on="Zonal_Railway", how="left")

# Fill gaps with mean
mean_delay = df_delay_feat_cast.agg(F.avg("delay_spike_freq")).first()[0]
mean_weather = df_weather_feat_cast.agg(F.avg("weather_correlation")).first()[0]

df_features = df_features.fillna({
    "delay_spike_freq":    round(mean_delay, 4),
    "weather_correlation": round(mean_weather, 4)
})

print(f"✅ Full feature table — {df_features.count()} zones")
df_features.orderBy("Zonal_Railway").show(25, truncate=False)

In [0]:
df_features.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.default.features_per_segment")

print("✅ features_per_segment written!")

spark.table("workspace.default.features_per_segment") \
    .orderBy("delay_spike_freq", ascending=False) \
    .show(25, truncate=False)